In [ ]:
!pip install leidenalg igraph -q

In [ ]:
import os, sys, pickle, json, time, traceback, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
CONFIG = {
    'emb_path': 'qwen35_4b_middle_embeddings.pkl',
    'save_dir': 'pipeline_checkpoints',
    'anisotropy_n_top': 12,
    'proto_min_occurrences': 4,
    'proto_max_senses': 6,
    'proto_stability_threshold': 0.70,
    'proto_n_bootstrap': 5,
    'graph_k': 30,
    'graph_min_shared': 8,
    'leiden_resolutions': [0.3, 0.6, 0.8, 1.2, 2.0],
    'leiden_main_resolution': 0.8,
    'n_distillation_iters': 3,
    'n_epochs': 60,
    'batch_size': 256,
    'lr_init': 3e-4,
    'w_neighbor_schedule': [0.35, 0.20, 0.10],
}

Path(CONFIG['save_dir']).mkdir(parents=True, exist_ok=True)
LOG_PATH = Path(CONFIG['save_dir']) / 'pipeline_log.json'

In [ ]:
CONFIG['graph_k']          = 100
CONFIG['graph_min_shared'] = 10
CONFIG['graph_min_weight'] = 0.06 
CONFIG['graph_fallback_weight'] = 0.03

In [ ]:
def _load_log():
    if LOG_PATH.exists():
        with open(LOG_PATH) as f: return json.load(f)
    return {'stages': {}}

def _save_log(log):
    with open(LOG_PATH, 'w') as f:
        json.dump(log, f, indent=2, ensure_ascii=False)

def is_done(stage_id):
    log = _load_log()
    return (str(stage_id) in log['stages']
            and log['stages'][str(stage_id)].get('status') == 'completed')

def mark_done(stage_id, name, files):
    log = _load_log()
    log['stages'][str(stage_id)] = {
        'name': name, 'status': 'completed',
        'timestamp': datetime.now().isoformat(), 'files': files,
    }
    _save_log(log)

def mark_failed(stage_id, error):
    log = _load_log()
    log['stages'][str(stage_id)] = {
        'status': 'failed',
        'timestamp': datetime.now().isoformat(),
        'error': str(error)[:300],
    }
    _save_log(log)

def invalidate_from(stage_id):
    log = _load_log()
    for k in list(log['stages']):
        if int(k) >= stage_id:
            log['stages'][k]['status'] = 'invalidated'
    _save_log(log)
    print(f"Этапы {stage_id}+ инвалидированы")

def stage_dir(stage_id):
    p = Path(CONFIG['save_dir']) / f'stage_{stage_id:02d}'
    p.mkdir(exist_ok=True)
    return p

def save_artifact(stage_id, name, obj):
    path = stage_dir(stage_id) / f'{name}.pkl'
    tmp  = stage_dir(stage_id) / f'{name}.pkl.tmp'
    with open(tmp, 'wb') as f: pickle.dump(obj, f, protocol=4)
    tmp.replace(path)
    return str(path)

def load_artifact(stage_id, name):
    path = stage_dir(stage_id) / f'{name}.pkl'
    with open(path, 'rb') as f: return pickle.load(f)

def load_stage(stage_id):
    d = stage_dir(stage_id)
    return {
        p.stem: pickle.load(open(p, 'rb'))
        for p in d.glob('*.pkl')
        if not p.name.endswith('.tmp')
    }

def show_status():
    log = _load_log()
    names = {
        0:'Загрузка', 1:'Anisotropy', 2:'Prototypes',
        3:'SNN граф', 4:'Leiden', 5:'Диагностика',
        6:'Metric Learning', 7:'Apply projector', 8:'Экспорт',
    }
    print('─'*52)
    for sid in range(9):
        info   = log['stages'].get(str(sid), {})
        status = info.get('status', 'pending')
        ts     = info.get('timestamp', '')[:16]
        sym    = {'completed':'✓','failed':'✗',
                  'invalidated':'○','pending':'·'}.get(status,'?')
        print(f'  [{sym}] {sid}: {names.get(sid,""):20s} {status} {ts}')
    print('─'*52)

show_status()

In [ ]:
# invalidate_from(3)
# sub = stage_dir(6) / 'sub_checkpoint.pkl'
# if sub.exists(): sub.unlink(); print("Удалён")

# Models

In [ ]:
class AnisotropyRemover:
    def __init__(self, n_top_components=12):
        self.n_top = n_top_components
        self.mean_vec = None
        self.top_components = None

    def fit(self, X):
        self.mean_vec = X.mean(axis=0)
        X_c = X - self.mean_vec
        try:
            from sklearn.utils.extmath import randomized_svd
            _, _, Vt = randomized_svd(X_c, n_components=self.n_top,
                                      random_state=42)
            self.top_components = Vt
        except Exception:
            _, _, Vt = np.linalg.svd(X_c, full_matrices=False)
            self.top_components = Vt[:self.n_top]
        return self

    def transform(self, X):
        X_c = X - self.mean_vec
        for comp in self.top_components:
            X_c = X_c - np.outer(X_c @ comp, comp)
        from sklearn.preprocessing import normalize
        return normalize(X_c, norm='l2')

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [ ]:
def _medoid(X):
    if len(X) == 1: return 0
    n = len(X)
    idx = np.random.choice(n, min(200, n), replace=False)
    X_s = X[idx]
    dists = 1 - X_s @ X_s.T
    np.fill_diagonal(dists, 0)
    return int(idx[np.argmin(dists.sum(axis=1))])

def _find_stable_senses(embs, max_k, n_bootstrap, stability_threshold):
    from sklearn.metrics import silhouette_score, normalized_mutual_info_score
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform

    n = len(embs)
    dists = np.clip(1 - embs @ embs.T, 0, 2)
    np.fill_diagonal(dists, 0)
    try:
        Z = linkage(squareform(dists, checks=False), method='average')
    except Exception:
        return 1, np.zeros(n, dtype=int)

    best_k, best_labels = 1, np.zeros(n, dtype=int)
    for k in range(2, min(max_k + 1, n)):
        labels = fcluster(Z, t=k, criterion='maxclust') - 1
        if len(set(labels)) < k: continue
        try:
            sil = silhouette_score(embs, labels, metric='cosine')
        except Exception:
            sil = 0.0
        if sil < 0.08: continue

        stab = []
        for _ in range(n_bootstrap):
            if n < 8: break
            s = max(int(n * 0.8), 4)
            bi = np.random.choice(n, s, replace=False)
            Xb = embs[bi]
            db = np.clip(1 - Xb @ Xb.T, 0, 2)
            np.fill_diagonal(db, 0)
            try:
                Zb = linkage(squareform(db, checks=False), method='average')
                lb = fcluster(Zb, t=k, criterion='maxclust') - 1
                stab.append(normalized_mutual_info_score(labels[bi], lb))
            except Exception:
                stab.append(0.0)

        if not stab: break
        if np.mean(stab) >= stability_threshold:
            best_k, best_labels = k, labels
        else:
            break
    return best_k, best_labels

In [ ]:
def build_stable_sense_prototypes(all_embeddings, anisotropy_remover,
                                   min_occurrences=4, max_senses=6,
                                   stability_threshold=0.70, n_bootstrap=5,
                                   n_anchors_per_sense=3):
    from sklearn.preprocessing import normalize as sk_normalize
    SKIP_UPOS = {'ADP','CONJ','PART','AUX','SCONJ','CCONJ','INTJ','PUNCT','SYM','X'}

    lemma_groups = defaultdict(list)
    for key, data in all_embeddings.items():
        if not isinstance(data, dict) or data.get('form') == '#NULL': continue
        if data.get('sem_class', '_') == '_': continue
        if data.get('upos', '') in SKIP_UPOS: continue
        lemma = data.get('lemma', data['form']).lower().strip()
        if len(lemma) < 2: continue
        emb = np.asarray(data['embedding'], dtype=np.float32)
        lemma_groups[lemma].append({
            'key': key, 'emb': emb,
            'sem_class': data.get('sem_class','_'),
            'upos': data.get('upos','')
        })

    prototypes, proto_meta, token_to_proto = [], [], {}

    for lemma, items in lemma_groups.items():
        n = len(items)
        keys = [x['key'] for x in items]
        raw  = np.stack([x['emb'] for x in items]).astype(np.float32)
        embs = anisotropy_remover.transform(raw)

        if n < min_occurrences:
            pi = len(prototypes)
            prototypes.append(embs[0])
            proto_meta.append({
                'lemma': lemma, 'sense_id': 0, 'n_tokens': n,
                'token_keys': keys, 'cohesion': 1.0,
                'sem_class_dist': Counter(x['sem_class'] for x in items)
            })
            for k in keys: token_to_proto[k] = pi
            continue

        best_k, sense_labels = _find_stable_senses(
            embs, max_senses, n_bootstrap, stability_threshold
        )

        for sid in range(best_k):
            mask = sense_labels == sid
            if not mask.any(): continue
            se   = embs[mask]
            skeys = [keys[i] for i in range(n) if mask[i]]
            sitems= [items[i] for i in range(n) if mask[i]]

            med = _medoid(se)
            pv  = se[med]

            anchors = [pv]
            if len(se) > n_anchors_per_sense:
                c = se.mean(axis=0); c /= np.linalg.norm(c) + 1e-8
                dfc = 1 - se @ c
                for ai in np.argsort(dfc)[-(n_anchors_per_sense-1):]:
                    anchors.append(se[ai])

            cohesion = float((se @ pv).mean()) if len(se) > 1 else 1.0

            pi = len(prototypes)
            prototypes.append(pv)
            proto_meta.append({
                'lemma': lemma, 'sense_id': sid,
                'n_tokens': int(mask.sum()), 'token_keys': skeys,
                'cohesion': cohesion, 'anchor_vecs': anchors,
                'sem_class_dist': Counter(x['sem_class'] for x in sitems)
            })
            for k in skeys: token_to_proto[k] = pi

    X_proto = np.stack(prototypes).astype(np.float32)
    print(f"Прототипов: {len(X_proto)}, лемм: {len(lemma_groups)}")
    return X_proto, proto_meta, token_to_proto

In [ ]:
class SemanticGraph:
    def __init__(self, k=30, min_shared_neighbors=8, min_edge_weight=0.0):
        self.k = k
        self.min_shared = min_shared_neighbors
        self.min_weight = min_edge_weight
        self.adjacency = None
        self._n = None
        self.knn_indices = None
        self.knn_distances = None

    def fit(self, X):
        from sklearn.neighbors import NearestNeighbors
        from scipy.sparse import csr_matrix

        n = len(X); self._n = n
        nn = NearestNeighbors(n_neighbors=self.k+1, metric='cosine',
                              algorithm='brute', n_jobs=-1)
        nn.fit(X)
        dists, idxs = nn.kneighbors(X)
        self.knn_distances = dists[:, 1:]
        self.knn_indices   = idxs[:, 1:]

        knn_sets = [set(row) for row in self.knn_indices]
        rows, cols, weights = [], [], []

        for i in range(n):
            si = self.knn_distances[i, -1] + 1e-8
            for pos, j in enumerate(self.knn_indices[i]):
                if i not in knn_sets[j]: continue
                shared = len(knn_sets[i] & knn_sets[j])
                if shared < self.min_shared: continue
                union  = len(knn_sets[i] | knn_sets[j])
                jacc   = shared / union
                sj     = self.knn_distances[j, -1] + 1e-8
                dij    = self.knn_distances[i, pos]
                kw     = float(np.exp(-dij**2 / (si * sj)))
                w      = jacc * kw
                if w < self.min_weight: continue
                rows.append(i); cols.append(j)
                weights.append(w)

        adj = csr_matrix((weights,(rows,cols)), shape=(n,n), dtype=np.float32)
        adj = (adj + adj.T) / 2.0

        # ── Fallback для изолированных узлов ─────────────────
        deg = np.array(adj.sum(axis=1)).ravel()
        isolated = np.where(deg == 0)[0]
        if len(isolated) > 0:
            print(f"  Fallback для {len(isolated)} изолированных...")
            fb_rows, fb_cols, fb_weights = [], [], []
            for i in isolated:
                if len(self.knn_indices[i]) == 0: continue
                # берём топ-3 ближайших соседей без фильтров
                for pos in range(min(3, len(self.knn_indices[i]))):
                    j   = self.knn_indices[i, pos]
                    dij = self.knn_distances[i, pos]
                    si  = self.knn_distances[i, -1] + 1e-8
                    sj  = self.knn_distances[j, -1] + 1e-8
                    kw  = float(np.exp(-dij**2 / (si * sj)))
                    fb_rows += [i, j]
                    fb_cols += [j, i]
                    fb_weights += [kw * 0.5, kw * 0.5]  # слабый вес

            if fb_rows:
                fb_adj = csr_matrix(
                    (fb_weights, (fb_rows, fb_cols)),
                    shape=(n, n), dtype=np.float32
                )
                adj = adj + fb_adj

        self.adjacency = adj

        deg_final = np.array(adj.sum(axis=1)).ravel()
        n_iso = (deg_final == 0).sum()
        print(f"Граф: {n} узлов, {len(rows)//2} рёбер основных, "
              f"изолированных после fallback: {n_iso}")
        return self

    def get_leiden_basins(self, resolution=0.8, n_iterations=10):
        try:
            import leidenalg, igraph as ig
        except ImportError:
            print("leidenalg недоступен => connected_components")
            from scipy.sparse.csgraph import connected_components
            _, labels = connected_components(self.adjacency, directed=False)
            return labels

        adj = self.adjacency.tocoo()
        filtered = [
            (s, t, w) for s, t, w
            in zip(adj.row.tolist(), adj.col.tolist(), adj.data.tolist())
            if s < t
        ]
        if not filtered:
            return np.zeros(self._n, dtype=int)

        srcs, tgts, wts = zip(*filtered)
        g = ig.Graph(n=self._n, edges=list(zip(srcs,tgts)), directed=False)
        g.es['weight'] = list(wts)

        part = leidenalg.find_partition(
            g, leidenalg.RBConfigurationVertexPartition,
            weights='weight', resolution_parameter=resolution,
            n_iterations=n_iterations, seed=42,
        )
        labels = np.array(part.membership, dtype=int)
        print(f"Leiden (res={resolution}): {len(set(labels))} сообществ")
        return labels

    def get_multi_resolution_basins(self, resolutions=None):
        if resolutions is None: resolutions = [0.3, 0.6, 0.8, 1.2, 2.0]
        return {r: self.get_leiden_basins(resolution=r) for r in resolutions}

    def normalized_cut(self, mask_a, mask_b):
        adj = self.adjacency.toarray()
        cut  = float(adj[np.ix_(mask_a, mask_b)].sum())
        va   = float(adj[mask_a].sum())
        vb   = float(adj[mask_b].sum())
        if va < 1e-8 or vb < 1e-8: return 0.0
        return cut/va + cut/vb

    def modularity_gain(self, mask_a, mask_b):
        adj = self.adjacency.toarray()
        tw  = adj.sum() / 2.0 + 1e-8
        eaa = adj[np.ix_(mask_a,mask_a)].sum()/2
        ebb = adj[np.ix_(mask_b,mask_b)].sum()/2
        eab = adj[np.ix_(mask_a,mask_b)].sum()
        ka  = adj[mask_a].sum(); kb = adj[mask_b].sum()
        qm  = (eaa+ebb+eab)/tw - ((ka+kb)/(2*tw))**2
        qs  = eaa/tw-(ka/(2*tw))**2 + ebb/tw-(kb/(2*tw))**2
        return float(qm - qs)

    def neighbor_entropy(self, labels):
        from scipy.stats import entropy as sp_entropy
        n = self.adjacency.shape[0]
        ul = np.unique(labels)
        l2i = {l: i for i, l in enumerate(ul)}
        ents = np.zeros(n)
        for i in range(n):
            nb = self.adjacency[i].indices
            if len(nb) == 0: continue
            cnts = np.bincount([l2i[labels[j]] for j in nb],
                               minlength=len(ul))
            probs = cnts / (cnts.sum() + 1e-10)
            ents[i] = float(sp_entropy(probs + 1e-10))
        return ents

In [ ]:
class GraphBasinMiner:
    def __init__(self, X, graph, proto_meta, sem_classes, basin_labels):
        self.X = X; self.graph = graph
        self.proto_meta = proto_meta
        self.classes = sem_classes; self.basins = basin_labels

        n = len(X)
        idx = np.random.choice(n, min(5000, n), replace=False)
        Xs  = X[idx]
        sims = (Xs @ Xs.T)[np.triu_indices(len(Xs), k=1)]
        
        p20 = float(np.percentile(sims, 20))
        p50 = float(np.percentile(sims, 50))
        p80 = float(np.percentile(sims, 80))
        
        # Если сходства близки к 0 — используем фиксированные пороги
        if p80 - p20 < 0.05:
            print(f"Вырожденное sim-пространство "
                  f"(p20={p20:.3f} p80={p80:.3f}) => фиксированные пороги")
            self._p20 = -0.05
            self._p50 =  0.10
            self._p80 =  0.25
            self._degenerate = True
        else:
            self._p20 = p20
            self._p50 = p50
            self._p80 = p80
            self._degenerate = False

        print(f"Sim thresholds: p20={self._p20:.3f} "
              f"p50={self._p50:.3f} p80={self._p80:.3f}")

    def mine(self, max_per_type=3000):
        basin2p = defaultdict(list); lemma2p = defaultdict(list)
        for i, (m, b) in enumerate(zip(self.proto_meta, self.basins)):
            basin2p[b].append(i); lemma2p[m['lemma']].append(i)

        pairs = []

        pc = 0
        for basin, members in basin2p.items():
            if len(members) < 2 or pc >= max_per_type: break
            for ii in range(len(members)):
                for jj in range(ii + 1, len(members)):
                    ia, ib = members[ii], members[jj]
                    if self.proto_meta[ia]['lemma'] == \
                       self.proto_meta[ib]['lemma']:
                        continue
                    sim = float(self.X[ia] @ self.X[ib])
                    if sim < self._p50: 
                        conf = 0.6 + 0.2 * min(len(members) / 50, 1.0)
                        pairs.append({
                            'idx_a': ia, 'idx_b': ib, 'type': 'pull',
                            'confidence': conf, 'current_sim': sim,
                            'source': 'basin_hard_pos'
                        })
                        pc += 1
                        if pc >= max_per_type: break
                if pc >= max_per_type: break

        pushc = 0
        for lemma, members in lemma2p.items():
            if len(members) < 2 or pushc >= max_per_type: break
            by_basin = defaultdict(list)
            for m in members: by_basin[self.basins[m]].append(m)
            if len(by_basin) < 2: continue
            bgs = list(by_basin.values())
            for gi in range(len(bgs)):
                for gj in range(gi + 1, len(bgs)):
                    for ia in bgs[gi]:
                        for ib in bgs[gj]:
                            sim  = float(self.X[ia] @ self.X[ib])
                            conf = min(
                                0.7 + 0.3 * max(0, self._p50 - sim)
                                / (abs(self._p50) + 1e-8), 1.0
                            )
                            pairs.append({
                                'idx_a': ia, 'idx_b': ib, 'type': 'push',
                                'confidence': conf, 'current_sim': sim,
                                'source': 'polysemy'
                            })
                            pushc += 1
        wc = 0
        for i in range(len(self.X)):
            if wc >= max_per_type: break
            for j in self.graph.adjacency[i].indices:
                if j <= i: continue
                if (self.classes[i] == self.classes[j]
                        and self.basins[i] == self.basins[j]):
                    sim = float(self.X[i] @ self.X[j])
                    # Условие расширено для вырожденного пространства
                    if self._p20 < sim < self._p80 or self._degenerate:
                        pairs.append({
                            'idx_a': i, 'idx_b': j, 'type': 'weak_pull',
                            'confidence': 0.4, 'current_sim': sim,
                            'source': 'class_basin_agree'
                        })
                        wc += 1

        pull_n = sum(1 for p in pairs if p['type'] == 'pull')
        push_n = sum(1 for p in pairs if p['type'] == 'push')
        weak_n = sum(1 for p in pairs if p['type'] == 'weak_pull')
        print(f"Пары: pull={pull_n}, push={push_n}, weak_pull={weak_n}")

        if pull_n + push_n + weak_n == 0:
            print(f"  basin2p: {len(basin2p)} бассейнов, "
                  f"размеры: {sorted([len(v) for v in basin2p.values()])[-5:]}")
            print(f"  lemma2p с полисемией: "
                  f"{sum(1 for v in lemma2p.values() if len(v)>1)}")

        return pairs
        
    def build_triplets(self, pairs, max_triplets=5000):
        import random
        pull_by_basin = defaultdict(list)
        for p in pairs:
            if p['type'] in ('pull','weak_pull'):
                pull_by_basin[int(self.basins[p['idx_a']])].append(p)

        push_pairs = [p for p in pairs if p['type']=='push']
        if not push_pairs: return []

        triplets = []
        for pp in push_pairs:
            if len(triplets)>=max_triplets: break
            ia,ib = pp['idx_a'],pp['idx_b']
            ba = int(self.basins[ia])
            cands = pull_by_basin.get(ba,[])
            if not cands: continue
            pos_p = random.choice(cands)
            anc = ia
            pos = pos_p['idx_b'] if pos_p['idx_a']==anc else pos_p['idx_a']
            neg = ib
            if anc==pos or anc==neg or pos==neg: continue
            triplets.append({
                'anchor':anc,'positive':pos,'negative':neg,
                'confidence':min(pp['confidence'],pos_p['confidence'])
            })
        print(f"Триплетов: {len(triplets)}")
        return triplets

In [ ]:
class TopologyProjector(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = dim // 2
        self.input_norm = nn.LayerNorm(dim)
        self.block1 = nn.Sequential(
            nn.Linear(dim,hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.05)
        )
        self.block2 = nn.Sequential(
            nn.Linear(hidden,hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        self.out_proj = nn.Linear(hidden, dim, bias=False)
        self.out_norm = nn.LayerNorm(dim)
        self.log_alpha = nn.Parameter(torch.tensor(2.2))

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.05)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        alpha = torch.sigmoid(self.log_alpha)
        xn = self.input_norm(x)
        h  = self.block2(self.block1(xn))
        delta = self.out_norm(self.out_proj(h))
        out = alpha * xn + (1 - alpha) * delta
        return F.normalize(out, dim=-1)

In [ ]:
class TopologyLoss(nn.Module):
    def __init__(self, margin=0.25, knn_k=10):
        super().__init__()
        self.margin = margin
        self.knn_k  = knn_k

    def triplet(self, anc, pos, neg, conf):
        sp = F.cosine_similarity(anc, pos, dim=-1)
        sn = F.cosine_similarity(anc, neg, dim=-1)
        return (conf * F.relu(sn - sp + self.margin)).mean()

    def pair_push(self, ea, eb, is_push, conf):
        sim = F.cosine_similarity(ea, eb, dim=-1)
        lp  = F.relu(sim - 0.15) * is_push
        ll  = F.relu(0.25 - sim) * (1 - is_push)
        return (conf * (lp + 0.4*ll)).mean()

    def neighborhood_kl(self, x_orig, x_proj, temperature=0.1):
        B = x_orig.shape[0]; k = min(self.knn_k, B-1)
        if k < 2: return torch.tensor(0.0, device=x_orig.device)
        eye = torch.eye(B, device=x_orig.device, dtype=torch.bool)
        so  = (x_orig @ x_orig.T).masked_fill(eye, float('-inf'))
        sp  = (x_proj @ x_proj.T).masked_fill(eye, float('-inf'))
        _, topk = so.topk(k, dim=-1)
        P = F.softmax(so.gather(1,topk)/temperature, dim=-1)
        Q = F.softmax(sp.gather(1,topk)/temperature, dim=-1)
        return F.kl_div(Q.log()+1e-10, P, reduction='batchmean')

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs, X, min_conf=0.3):
        self.X = X
        self.data = [
            (p['idx_a'], p['idx_b'],
             float(p['type']=='push'), float(p['confidence']))
            for p in pairs if p['confidence'] >= min_conf
        ]
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        ia,ib,isp,cf = self.data[i]
        return (torch.FloatTensor(self.X[ia]),
                torch.FloatTensor(self.X[ib]),
                torch.tensor(isp), torch.tensor(cf))

class TripletDataset(Dataset):
    def __init__(self, triplets, X):
        self.X = X; self.data = triplets
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        t = self.data[i]
        return (torch.FloatTensor(self.X[t['anchor']]),
                torch.FloatTensor(self.X[t['positive']]),
                torch.FloatTensor(self.X[t['negative']]),
                torch.tensor(float(t['confidence'])))

# Work

In [ ]:
STAGE = 0
if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s0 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Загрузка эмбеддингов...")
    t0 = time.time()
    try:
        with open(CONFIG['emb_path'], 'rb') as f:
            all_embeddings = pickle.load(f)

        valid_keys, valid_embs, valid_classes = [], [], []
        skipped = 0
        for key, data in all_embeddings.items():
            if not isinstance(data, dict): skipped+=1; continue
            if data.get('form')=='#NULL' or data.get('sem_class')=='_':
                skipped+=1; continue
            emb = np.asarray(data['embedding'], dtype=np.float32)
            if not np.isfinite(emb).all(): skipped+=1; continue
            valid_keys.append(key)
            valid_embs.append(emb)
            valid_classes.append(data.get('sem_class','_'))

        X_raw = np.stack(valid_embs).astype(np.float32)
        valid_classes_arr = np.array(valid_classes)
        stats = {'n_tokens':len(X_raw), 'dim':int(X_raw.shape[1]),
                 'n_classes':int(len(np.unique(valid_classes_arr))),
                 'n_skipped':skipped}

        files = []
        files.append(save_artifact(STAGE,'X_raw',X_raw))
        files.append(save_artifact(STAGE,'valid_keys',valid_keys))
        files.append(save_artifact(STAGE,'valid_classes',valid_classes_arr))
        files.append(save_artifact(STAGE,'stats',stats))
        files.append(save_artifact(STAGE,'all_embeddings',all_embeddings))
        mark_done(STAGE,'Загрузка',files)

        s0 = {'X_raw':X_raw,'valid_keys':valid_keys,
              'valid_classes':valid_classes_arr,
              'stats':stats,'all_embeddings':all_embeddings}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

print(f"  Токенов: {s0['stats']['n_tokens']}")
print(f"  Размерность: {s0['stats']['dim']}")
print(f"  Классов: {s0['stats']['n_classes']}")

In [ ]:
STAGE = 1
if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s1 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Anisotropy removal...")
    t0 = time.time()
    try:
        remover = AnisotropyRemover(CONFIG['anisotropy_n_top'])
        X_clean = remover.fit_transform(s0['X_raw'])

        files = [save_artifact(STAGE,'X_clean',X_clean),
                 save_artifact(STAGE,'remover',remover)]
        mark_done(STAGE,'Anisotropy',files)
        s1 = {'X_clean':X_clean,'remover':remover}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

print(f"  Shape: {s1['X_clean'].shape}")

In [ ]:
STAGE = 2
if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s2 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Sense prototypes...")
    t0 = time.time()
    try:
        X_proto, proto_meta, token_to_proto = build_stable_sense_prototypes(
            s0['all_embeddings'], s1['remover'],
            min_occurrences=CONFIG['proto_min_occurrences'],
            max_senses=CONFIG['proto_max_senses'],
            stability_threshold=CONFIG['proto_stability_threshold'],
            n_bootstrap=CONFIG['proto_n_bootstrap'],
            n_anchors_per_sense=3,
        )
        proto_classes = np.array([
            m['sem_class_dist'].most_common(1)[0][0]
            if m['sem_class_dist'] else 'UNKNOWN'
            for m in proto_meta
        ])

        files = [save_artifact(STAGE,'X_proto',X_proto),
                 save_artifact(STAGE,'proto_meta',proto_meta),
                 save_artifact(STAGE,'token_to_proto',token_to_proto),
                 save_artifact(STAGE,'proto_classes',proto_classes)]
        mark_done(STAGE,'Prototypes',files)
        s2 = {'X_proto':X_proto,'proto_meta':proto_meta,
              'token_to_proto':token_to_proto,'proto_classes':proto_classes}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

print(f"  Прототипов: {len(s2['X_proto'])}")
print(f"  Токенов с маппингом: {len(s2['token_to_proto'])}")

In [ ]:
STAGE = 3
if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s3 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Mutual SNN граф...")
    t0 = time.time()
    try:
        graph = SemanticGraph(CONFIG['graph_k'], CONFIG['graph_min_shared'])
        graph.fit(s2['X_proto'])

        adj = graph.adjacency
        deg = np.array(adj.sum(axis=1)).ravel()
        graph_stats = {
            'n_nodes':adj.shape[0], 'n_edges':int(adj.nnz//2),
            'mean_degree':float(deg.mean()), 'max_degree':float(deg.max()),
            'n_isolated':int((deg==0).sum()),
        }

        files = [save_artifact(STAGE,'graph',graph),
                 save_artifact(STAGE,'graph_stats',graph_stats)]
        mark_done(STAGE,'SNN граф',files)
        s3 = {'graph':graph,'graph_stats':graph_stats}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

gs = s3['graph_stats']
print(f"  Узлов: {gs['n_nodes']}, рёбер: {gs['n_edges']}, "
      f"ср.степень: {gs['mean_degree']:.1f}")

In [ ]:
STAGE = 4
if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s4 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Leiden communities...")
    t0 = time.time()
    try:
        graph = s3['graph']
        multi_basins = graph.get_multi_resolution_basins(
            CONFIG['leiden_resolutions']
        )
        main_r = CONFIG['leiden_main_resolution']
        basin_labels = multi_basins[main_r]
        leiden_stats = {r: int(len(np.unique(lb)))
                        for r, lb in multi_basins.items()}

        files = [save_artifact(STAGE,'basin_labels',basin_labels),
                 save_artifact(STAGE,'multi_resolution_basins',multi_basins),
                 save_artifact(STAGE,'main_resolution',main_r),
                 save_artifact(STAGE,'leiden_stats',leiden_stats)]
        mark_done(STAGE,'Leiden',files)
        s4 = {'basin_labels':basin_labels,'multi_resolution_basins':multi_basins,
              'main_resolution':main_r,'leiden_stats':leiden_stats}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

for r,n in s4['leiden_stats'].items():
    marker = " ← основной" if r==s4['main_resolution'] else ""
    print(f"  {r}: {n} сообществ{marker}")

In [ ]:
import numpy as np
from collections import Counter

labels = s4['basin_labels']
unique, counts = np.unique(labels, return_counts=True)
basin_counter = Counter(counts)

print("Размер сообществ:")
print(f"  Всего бассейнов: {len(unique)}")
print(f"  Общий размер топ-10: {sum(counts[:10])} прототипов ({sum(counts[:10])/len(labels)*100:.1f}%)")
print(f"  Топ-10 размеров: {[int(x) for x in counts[:10]]}")
print(f"  Бассейнов размера 1: {sum(counts==1)}")
print(f"  Размер >100: {sum(counts>100)}")
print(f"  Средний размер: {counts.mean():.0f}")
print(f"  Медианный размер: {np.median(counts):.0f}")
print(f"  Макс размер: {counts.max()}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

X = s2['X_proto']
print(f"Тестируем параметры графа на X_proto...")

for k, min_sh in [(50, 2), (100, 2), (50, 1), (100, 1)]:
    nn = NearestNeighbors(
        n_neighbors=min(k+1, len(X)), 
        metric='cosine', algorithm='brute', n_jobs=-1
    )
    nn.fit(X)
    dists, idxs = nn.kneighbors(X[:2000])
    
    knn_sets = [set(row[1:]) for row in idxs]
    edges = 0
    isolated = 0
    for i in range(2000):
        has_edge = False
        for j in knn_sets[i]:
            if j < 2000 and i in knn_sets[j]:
                shared = len(knn_sets[i] & knn_sets[j])
                if shared >= min_sh:
                    edges += 1
                    has_edge = True
        if not has_edge:
            isolated += 1
    
    print(f"  k={k:3d} min_shared={min_sh}: "
          f"рёбер={edges:6d}, "
          f"изолированных={isolated:4d} ({isolated/2000*100:.1f}%)"
          f" → ср.степень≈{edges*2/2000:.1f}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
from scipy.sparse import csr_matrix

X = s2['X_proto']
n = len(X)

nn = NearestNeighbors(n_neighbors=101, metric='cosine',
                      algorithm='brute', n_jobs=-1)
nn.fit(X)
dists, idxs = nn.kneighbors(X)
knn_sets = [set(row[1:]) for row in idxs]

for min_sh in [5, 8, 10, 15, 20]:
    edges = 0
    isolated = 0
    for i in range(n):
        has_edge = False
        for j in idxs[i][1:]:
            if i in knn_sets[j]:
                shared = len(knn_sets[i] & knn_sets[j])
                if shared >= min_sh:
                    edges += 1
                    has_edge = True
        if not has_edge:
            isolated += 1
    print(f"  min_shared={min_sh:2d}: "
          f"рёбер={edges//2:7d}, "
          f"изолированных={isolated:5d} ({isolated/n*100:.1f}%), "
          f"ср.степень≈{edges/n:.1f}")

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

X = s2['X_proto']
n = len(X)

nn = NearestNeighbors(n_neighbors=101, metric='cosine',
                      algorithm='brute', n_jobs=-1)
nn.fit(X)
dists, idxs = nn.kneighbors(X)
knn_sets = [set(row[1:]) for row in idxs]

min_sh = 15  # фиксируем

for weight_thr in [0.01, 0.05, 0.10, 0.15, 0.20]:
    edges = 0
    isolated = 0
    for i in range(n):
        has_edge = False
        si = dists[i, -1] + 1e-8
        for pos, j in enumerate(idxs[i][1:]):
            if i not in knn_sets[j]:
                continue
            shared = len(knn_sets[i] & knn_sets[j])
            if shared < min_sh:
                continue
            union  = len(knn_sets[i] | knn_sets[j])
            jacc   = shared / union
            sj     = dists[j, -1] + 1e-8
            dij    = dists[i, pos]
            kw     = float(np.exp(-dij**2 / (si * sj)))
            w      = jacc * kw
            if w >= weight_thr:
                edges += 1
                has_edge = True
        if not has_edge:
            isolated += 1

    print(f"  min_shared={min_sh} weight>={weight_thr:.2f}: "
          f"рёбер={edges//2:7d}, "
          f"изолированных={isolated:5d} ({isolated/n*100:.1f}%), "
          f"ср.степень≈{edges/n:.1f}")

In [ ]:
# Тестируем min_shared=10 с разными весами
min_sh = 10

for weight_thr in [0.06, 0.07, 0.08, 0.09]:
    edges = 0
    isolated = 0
    for i in range(n):
        has_edge = False
        si = dists[i, -1] + 1e-8
        for pos, j in enumerate(idxs[i][1:]):
            if i not in knn_sets[j]:
                continue
            shared = len(knn_sets[i] & knn_sets[j])
            if shared < min_sh:
                continue
            union  = len(knn_sets[i] | knn_sets[j])
            jacc   = shared / union
            sj     = dists[j, -1] + 1e-8
            dij    = dists[i, pos]
            kw     = float(np.exp(-dij**2 / (si * sj)))
            w      = jacc * kw
            if w >= weight_thr:
                edges += 1
                has_edge = True
        if not has_edge:
            isolated += 1

    print(f"  min_shared={min_sh} weight>={weight_thr:.2f}: "
          f"рёбер={edges//2:7d}, "
          f"изолированных={isolated:5d} ({isolated/n*100:.1f}%), "
          f"ср.степень≈{edges/n:.1f}")

In [ ]:
# STAGE = 5
# if is_done(STAGE):
#     print(f"[{STAGE}] Загружаем из чекпоинта...")
#     s5 = load_stage(STAGE)
# else:
#     print(f"[{STAGE}] Диагностика...")
#     t0 = time.time()
#     try:
#         import pandas as pd
#         diag = GeometryDiagnostics(
#             s2['X_proto'], s2['proto_classes'], s3['graph']
#         )
#         df_compact  = diag.class_compactness(min_size=20)
#         df_merge    = diag.merge_candidates(df_compact)
#         df_split    = diag.split_candidates(df_compact, s0['all_embeddings'])
#         df_boundary = diag.boundary_analysis()

#         sp = Path(CONFIG['save_dir'])
#         df_compact.drop(columns=['centroid'],errors='ignore').to_csv(
#             sp/'class_geometry_stats.csv', index=False, encoding='utf-8-sig'
#         )
#         with pd.ExcelWriter(sp/'annotation_tasks.xlsx',
#                             engine='openpyxl') as writer:
#             m=df_merge.copy(); m['your_decision']=''; m['comment']=''
#             m.to_excel(writer, sheet_name='1_MERGE', index=False)
#             s=df_split.copy()
#             s['your_decision']=''; s['split_into']=''; s['comment']=''
#             s.to_excel(writer, sheet_name='2_SPLIT', index=False)
#             b=df_boundary.copy(); b['your_decision']=''; b['comment']=''
#             b.to_excel(writer, sheet_name='3_BOUNDARY', index=False)

#         files = [save_artifact(STAGE,'df_compact',df_compact),
#                  save_artifact(STAGE,'df_merge',df_merge),
#                  save_artifact(STAGE,'df_split',df_split),
#                  save_artifact(STAGE,'df_boundary',df_boundary)]
#         mark_done(STAGE,'Диагностика',files)
#         s5 = {'df_compact':df_compact,'df_merge':df_merge,
#               'df_split':df_split,'df_boundary':df_boundary}
#         print(f"  Время: {time.time()-t0:.1f}с")
#     except Exception as e:
#         mark_failed(STAGE, str(e)); raise

# print(f"  MERGE: {len(s5['df_merge'])}, SPLIT: {len(s5['df_split'])}, "
#       f"BOUNDARY: {len(s5['df_boundary'])}")
# print(f"\nТоп MERGE:")
# print(s5['df_merge'][['class_a','class_b','centroid_sim',
#                         'recommendation']].head(5).to_string(index=False))
# print(f"\nТоп SPLIT:")
# print(s5['df_split'][['sem_class','split_strength',
#                         'recommendation']].head(5).to_string(index=False))

In [ ]:
STAGE = 6
SUB_CKPT = stage_dir(STAGE) / 'sub_checkpoint.pkl'

def _save_sub(X_cur, model, log, next_it):
    tmp = stage_dir(STAGE) / 'sub_checkpoint.pkl.tmp'
    with open(tmp,'wb') as f:
        pickle.dump({'X_current':X_cur,
                     'model_state':{k:v.cpu()
                                    for k,v in model.state_dict().items()},
                     'log':log, 'next_iter':next_it}, f, protocol=4)
    tmp.replace(SUB_CKPT)

if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s6 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Metric Learning...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Устройство: {device}")

    n_iters  = CONFIG['n_distillation_iters']
    n_epochs = CONFIG['n_epochs']
    bs       = CONFIG['batch_size']
    lr_init  = CONFIG['lr_init']
    w_nb_sch = list(CONFIG['w_neighbor_schedule'])
    while len(w_nb_sch) < n_iters: w_nb_sch.append(0.05)

    dim = s2['X_proto'].shape[1]

    if SUB_CKPT.exists():
        sub = pickle.load(open(SUB_CKPT,'rb'))
        X_current  = sub['X_current']
        train_log  = sub['log']
        start_iter = sub['next_iter']
        model = TopologyProjector(dim).to(device)
        model.load_state_dict(
            {k:v.to(device) for k,v in sub['model_state'].items()}
        )
        print(f"  Sub-checkpoint: продолжаем с итерации {start_iter+1}/{n_iters}")
    else:
        X_current  = s2['X_proto'].copy()
        train_log  = []
        start_iter = 0
        model      = None

    try:
        for it in range(start_iter, n_iters):
            print(f"\n  Итерация {it+1}/{n_iters} | "
                  f"w_nb={w_nb_sch[it]:.2f} | "
                  f"lr={lr_init*(0.5**it):.2e}")
            t_it = time.time()

            g_it = SemanticGraph(CONFIG['graph_k'], CONFIG['graph_min_shared'])
            g_it.fit(X_current)
            basins_it = g_it.get_leiden_basins(CONFIG['leiden_main_resolution'])

            miner  = GraphBasinMiner(X_current, g_it, s2['proto_meta'],
                                     s2['proto_classes'], basins_it)
            pairs    = miner.mine(max_per_type=2000)
            triplets = miner.build_triplets(pairs, max_triplets=4000)
            if not pairs:
                print("  Нет пар — пропуск")
                continue

            pair_loader = DataLoader(
                PairDataset(pairs, X_current, min_conf=0.3),
                batch_size=bs, shuffle=True, drop_last=True
            )
            trip_loader = (DataLoader(
                TripletDataset(triplets, X_current),
                batch_size=bs//2, shuffle=True, drop_last=True)
                if triplets else None)

            if model is None:
                model = TopologyProjector(dim).to(device)

            opt  = torch.optim.AdamW(model.parameters(),
                                     lr=lr_init*(0.5**it), weight_decay=1e-5)
            crit = TopologyLoss(margin=0.25, knn_k=min(10,bs-1))
            sch  = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=n_epochs, eta_min=1e-6)

            w_nb = w_nb_sch[it]; tw = 0.50+0.30+w_nb
            w_t,w_p,w_n = 0.50/tw, 0.30/tw, w_nb/tw

            model.train()
            for epoch in range(n_epochs):
                ep = {k:0.0 for k in ('trip','pair','nb','total')}
                nb = 0; tri = iter(trip_loader) if trip_loader else None

                for ea,eb,isp,cf in pair_loader:
                    ea=ea.to(device); eb=eb.to(device)
                    isp=isp.to(device); cf=cf.to(device)
                    pa=model(ea); pb=model(eb)

                    lp = crit.pair_push(pa,pb,isp,cf)
                    ln = crit.neighborhood_kl(F.normalize(ea,dim=-1),pa)
                    lt = torch.tensor(0.0,device=device)
                    if tri:
                        try:
                            a,p,n,ct = [x.to(device) for x in next(tri)]
                            lt = crit.triplet(model(a),model(p),model(n),ct)
                        except StopIteration:
                            tri = iter(trip_loader) if trip_loader else None

                    loss = w_t*lt + w_p*lp + w_n*ln
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    opt.step()
                    ep['trip']+=lt.item(); ep['pair']+=lp.item()
                    ep['nb']+=ln.item(); ep['total']+=loss.item(); nb+=1

                sch.step()
                if (epoch+1)%20==0:
                    print(f"    ep {epoch+1:3d} | "
                          f"trip={ep['trip']/nb:.4f} | "
                          f"pair={ep['pair']/nb:.4f} | "
                          f"nb={ep['nb']/nb:.4f}")

            model.eval()
            chunks=[]
            with torch.no_grad():
                for i in range(0,len(X_current),512):
                    b=torch.FloatTensor(X_current[i:i+512]).to(device)
                    chunks.append(model(b).cpu().numpy())
            X_current = np.vstack(chunks)

            train_log.append({'iteration':it,'n_pairs':len(pairs),
                              'n_triplets':len(triplets),
                              'elapsed_s':round(time.time()-t_it,1)})
            _save_sub(X_current, model, train_log, it+1)
            print(f"  Итерация {it+1} завершена за "
                  f"{time.time()-t_it:.0f}с | sub-checkpoint сохранён")

        mst = {k:v.cpu() for k,v in model.state_dict().items()}
        files = [save_artifact(STAGE,'model_state',mst),
                 save_artifact(STAGE,'X_proto_refined',X_current),
                 save_artifact(STAGE,'training_log',train_log),
                 save_artifact(STAGE,'input_dim',dim)]
        mark_done(STAGE,'Metric Learning',files)
        if SUB_CKPT.exists(): SUB_CKPT.unlink()
        s6 = {'model_state':mst,'X_proto_refined':X_current,
              'training_log':train_log,'input_dim':dim}

    except KeyboardInterrupt:
        print("\n  Прерывание. Sub-checkpoint сохранён. "
              "Перезапустите ячейку для продолжения.")
        raise
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

print(f"  Итераций: {len(s6['training_log'])}")
for e in s6['training_log']:
    print(f"    iter {e['iteration']+1}: pairs={e['n_pairs']}, "
          f"triplets={e['n_triplets']}, t={e['elapsed_s']}s")

In [ ]:
STAGE = 7
if is_done(STAGE):
    print(f"[{STAGE}] Загружаем из чекпоинта...")
    s7 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Применение проектора...")
    t0 = time.time()
    try:
        t2p = s2['token_to_proto']
        Xr  = s6['X_proto_refined']
        refined = {}
        covered = 0; not_covered = 0

        for key, data in s0['all_embeddings'].items():
            if not isinstance(data, dict):
                refined[key] = data; continue
            pi = t2p.get(key)
            if pi is not None and pi < len(Xr):
                nd = dict(data)
                nd['embedding_original'] = data.get('embedding')
                nd['embedding_refined']  = Xr[pi].tolist()
                refined[key] = nd; covered += 1
            else:
                refined[key] = data; not_covered += 1

        cov = {'covered':covered,'not_covered':not_covered}
        files = [save_artifact(STAGE,'refined_embeddings',refined),
                 save_artifact(STAGE,'coverage',cov)]
        mark_done(STAGE,'Apply projector',files)
        s7 = {'refined_embeddings':refined,'coverage':cov}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

print(f"  Покрыто: {s7['coverage']['covered']}")
print(f"  Без маппинга: {s7['coverage']['not_covered']}")

In [ ]:
STAGE = 8
if is_done(STAGE):
    print(f"[{STAGE}] Уже выполнен")
    s8 = load_stage(STAGE)
else:
    print(f"[{STAGE}] Финальный экспорт...")
    t0 = time.time()
    try:
        sp = Path(CONFIG['save_dir'])

        torch.save(s6['model_state'], sp/'topology_projector_v2.pt')

        for fname in ('refined_embeddings','pipeline_artifacts_v2'):
            if fname == 'refined_embeddings':
                obj = s7['refined_embeddings']
            else:
                obj = {
                    'X_proto_refined':s6['X_proto_refined'],
                    'proto_meta':s2['proto_meta'],
                    'token_to_proto':s2['token_to_proto'],
                    'basin_labels':s4['basin_labels'],
                    'multi_resolution_basins':s4['multi_resolution_basins'],
                    'main_resolution':s4['main_resolution'],
                    'graph_adjacency':s3['graph'].adjacency,
                    'training_log':s6['training_log'],
                    'config':CONFIG,
                }
            tmp = sp / f'{fname}.pkl.tmp'
            with open(tmp,'wb') as f: pickle.dump(obj,f,protocol=4)
            tmp.replace(sp/f'{fname}.pkl')

        summary = {
            'n_proto':len(s2['X_proto']),
            'n_refined':len(s6['X_proto_refined']),
            'embedding_dim':s6['input_dim'],
            'n_basins':int(len(np.unique(s4['basin_labels']))),
            'training_iters':len(s6['training_log']),
            'coverage':s7['coverage'],
            'timestamp':datetime.now().isoformat(),
        }
        with open(sp/'pipeline_summary.json','w') as f:
            json.dump(summary,f,indent=2,ensure_ascii=False)

        files = [save_artifact(STAGE,'summary',summary)]
        mark_done(STAGE,'Экспорт',files)
        s8 = {'summary':summary}
        print(f"  Время: {time.time()-t0:.1f}с")
    except Exception as e:
        mark_failed(STAGE, str(e)); raise

sm = s8['summary']
print(f"\n{'='*50}")
print(f"ПАЙПЛАЙН ЗАВЕРШЁН")
print(f"  Прототипов:     {sm['n_proto']}")
print(f"  Dim:            {sm['embedding_dim']}")
print(f"  Leiden basins:  {sm['n_basins']}")
print(f"  Итераций:       {sm['training_iters']}")
print(f"  Покрытие:       {sm['coverage']['covered']}")